In [1]:
from sage.combinat.tuple import Tuples
import itertools
import numpy as np

# Defining the Klein-4 group

K = [[0,0], [0,1], [1,0], [1,1]]

# Defining addition of elements in the Klein-4 group K. 

def s(i, j):
    return [(i[0] + j[0]) %2, (i[1] + j[1]) %2]

"""
Our goal is to compute normalized 2-cocycles Z2(K, C2) for the trivial action of K acting on the cyclic group C2 with 2 elements.  The 2-cocycles are functions functions f:KxK --> {0, 1} satisfying certain conditions, and thus we first product all such functions.  We can store these functions as a list of length 16, where we think of the elements of KxK being ordered under the lexicographic ordering from 0,0 to 1, 1, and where the list corresponding to a 2-cocycle f:KxK --> {0, 1} has the value f(i, j) at the position corresponding to the element (i, j) of K. 
Specifically, we form a list G consisting of all pair KxK, and we will store a function as a list of 16 elements where 
f[G.index([h, k])] is the value of f at (h, k).  
"""


G = [] 

for x in K:
    for y in K:
        G.append([x, y])
        

my_elements = [0, 1]

# The following generates all tuples of length 4 with repetition.
all_tuples = list(itertools.product(my_elements, repeat=16))

all_tuples = [list(x)  for x in all_tuples]

'''
Next, we form the set of all cocycles, which are functions satisfying f(h, k) + f(g, hk) = f(gh, k) + f(g, h).
(Recall that the action of K on C2 is trivial). 
'''

cocycles = []

for f in all_tuples:
    add = True 
    for g in K:
        for h in K:
            for k in K:
                if ((f[G.index([h, k])])+(f[G.index([g, s(h, k)])]))%2 != ((f[G.index([s(g, h), k])])+(f[G.index([g,h])]))%2:
                    add = False
                    break 
            if not add:
                break
        if not add:
            break
    if add:
        cocycles.append(f)
        
# We only need to deal with normalized cocycles, and the below code filters out those cocycles which are not normalized.  
        
n_cocycles = []

for f in cocycles:
    add = True
    for g in K:
        if (f[G.index([[0,0], g])] != 0) or (f[G.index([g, [0, 0]])] != 0):
            add = False
    if add:
        n_cocycles.append(f)
        
        
"""
Now we want to form the list n_Imd which is the image of normalized 1-cocycles and we want to mod out by the relation

dc(g, h) = c(h) - c(gh) + c(g) (recalling that the action of K on C2 is trivial).  

As before, we begin by constructing all functions f:K --> {0, 1} as 4-tuples populated by 0, 1, and compute the possible distinct images under the differential.   
"""

one_tuples = list(itertools.product(my_elements, repeat=4))

one_tuples = [list(x)  for x in all_tuples]

Imd = [] 

for c in one_tuples:
    f = []
    for g in K:
        for h in K:
            f.append((c[K.index(h)]+c[K.index(s(g, h))]+c[K.index(g)])%2)
    if f not in Imd:
        Imd.append(f)
 
n_Imd = [Imd[0], Imd[1]]

"""
Now that we have determined the distinct images of the 1-cocycles, we can determine which of the normalized 2-cocycles are equivalent, and choose a single representative for each equivalence class.  The list called 'reduced' is such a list of representatives. 
"""

reduced = []

for f in n_cocycles:
    b = []
    for i in range(16):
        b.append((f[i]+n_Imd[1][i])%2)
    if (f not in reduced) and (b not in reduced):
        reduced.append(f)
        
        
        
'''
The following returns two lists, abelian and non_abelian, being populated by those 2-cocycles whose corresponding extensions are respectively abelian and non-abelian. 
'''

abelian = []

for f in reduced:
    add_to = True
    for i in K:
        for j in K:
            if f[G.index([i, j])] != f[G.index([j, i])]:
                add_to = False
                break
        if not add_to:
            break
    if add_to:
        abelian.append(f)
        
abelian

non_abelian = [] 

for f in reduced:
    if f not in abelian:
        non_abelian.append(f)

In [2]:
'''
The main goal of this cell is to confirm that all of the representations defined in the proof of Theorem... are well-defined and irreducible.  The output of this cell confirms that for all 4 possible extension types, this is the case. 

First, we will compose all the representatives of the 2-cocycles with the non trivial character of C2 to get functions KxK --> C_2 --> {1, -1}.  Like before, we represent them as 16-tuples valued in +/- 1.  
The list 'reduced_one' returns the compositions with the non-Abelian functions.    
'''

reduced_one = []

for f in non_abelian:
    new = []
    for x in f:
        if x == 0:
            new.append(1)
        else:
            new.append(-1)
    reduced_one.append(new)
    
'''
Below we create a list sigma_defined which adds those cocycles for which the representation sigma, on the corresponding extension, as outlined in the paper, is well-defined.  As one can see, the list has length 4, implying that the representation is well-defined for each extension. 

Note that we also test to see if sqrt(a) == - sqrt(b*c), for so long as it is not, the representation will be irreducible. 

Below, we define the values a, b, c, x, y, z which appear in the paper in the definition for the matrices of the representation sigma.
'''

sigma_defined = []

for f in reduced_one:
    add = True 
    
    a = f[G.index([[0, 1], [0, 1]])] #0
    b = f[G.index([[1, 0], [0, 1]])] #1
    c = f[G.index([[1, 1], [0, 1]])] #2

    x = f[G.index([[0, 1], [1, 0]])] #3
    y = f[G.index([[1, 0], [1, 0]])] #4
    z = f[G.index([[1, 1], [1, 0]])] #5
    # f_matrices.append([a, b, c, x, y, z])
    
    if sqrt(a) == - sqrt(b*c):
        add = False
    
    """
    Below we define the matrices defining the representation sigma.    
    """

    zz = Matrix([
        [1, 0],
        [0, 1]
    ])

    zo = Matrix([
        [sqrt(a), 0],
        [0, -sqrt(b*c)]
    ])

    oz = (a*b*c)* (Matrix([
        [0, x],
        [z, 0]
    ])) 
    
    oo = (f[G.index([[0, 1], [1, 0]])]*(Matrix([
        [1, 0],
        [0, 1]]
    )))*zo*oz
    
    K_matrices = [zz, zo, oz, oo]
    
    for g in K:
        for h in K:
            M = Matrix([[f[G.index([g, h])],0], [0, f[G.index([g, h])]]])
            if K_matrices[K.index(g)]*K_matrices[K.index(h)] != M*K_matrices[K.index(s(g, h))]:
                add = False                                              
                break
        if not add:
            break
            
    if add:
        sigma_defined.append(f)
        
        
print(len(sigma_defined))

4


In [3]:
"""
The following code computes the value of the Schur indicator of sigma for each of the extensions.  Observe that all bu the 4th one has Schur indicator 1.   
"""


diagonals = []

for f in reduced_one:
    values = []
    for g in K:
        values.append(f[G.index([g, g])])
    diagonals.append(values)
    
print(diagonals)

[[1, 1, 1, -1], [1, 1, -1, 1], [1, -1, 1, 1], [1, -1, -1, -1]]


In [4]:
"""
The list extra_functions below produces the composition of the non-trivial character on the two first factors of  C_2^k with the cocycle, which is to say, the value of the character chi, chi, 1, 1, ..., 1 on  C_2^k.   We then use this to produce a list sigma_two_defined which shows that the representation sigma is well-defined and irreducible for each of these extensions. Indeed, the printout of the length of the list below confirms that each of these representations is well-defined and irreducible, and according to the paper they have Schur indicator 1. 
"""

extra_functions = []

for i in range(1, len(abelian)):
    new = []
    for j in range(len(abelian[i])):
        if abelian[i][j] == 0:
            new.append(reduced_one[3][j])
        else:
            new.append(-reduced_one[3][j])
    extra_functions.append(new)
    
sigma_two_defined = []
    
#f_matrices = []

for f in extra_functions:
    add = True 
    
    a = f[G.index([[0, 1], [0, 1]])] #0
    b = f[G.index([[1, 0], [0, 1]])] #1
    c = f[G.index([[1, 1], [0, 1]])] #2

    x = f[G.index([[0, 1], [1, 0]])] #3
    y = f[G.index([[1, 0], [1, 0]])] #4
    z = f[G.index([[1, 1], [1, 0]])] #5
    # f_matrices.append([a, b, c, x, y, z])
    
    if sqrt(a) == - sqrt(b*c):
        add = False
    
    """
    Below we define the matrices defining the representation.  
    """

    zz = Matrix([
        [1, 0],
        [0, 1]
    ])

    zo = Matrix([
        [sqrt(a), 0],
        [0, -sqrt(b*c)]
    ])

    oz = (a*b*c)* (Matrix([
        [0, x],
        [z, 0]
    ])) 
    
    oo = (f[G.index([[0, 1], [1, 0]])]*(Matrix([
        [1, 0],
        [0, 1]]
    )))*zo*oz
    
    K_matrices = [zz, zo, oz, oo]
    
    for g in K:
        for h in K:
            M = Matrix([[f[G.index([g, h])],0], [0, f[G.index([g, h])]]])
            if K_matrices[K.index(g)]*K_matrices[K.index(h)] != M*K_matrices[K.index(s(g, h))]:
                add = False                                              
                break
        if not add:
            break
            
    if add:
        sigma_two_defined.append(f)
        
        
print(len(sigma_two_defined))

3


In [5]:
"""
Finally, we construct the group EE below which is supposed to be the V4 extension defined by non_abelian[3].
The printout verifies that there is only one element of order 2, hence EE is the group Q8.  
"""

EE = [] 

for i in [0, 1]:
    for k in K:
        EE.append([i, k])
        
def t(i, j):
    return([(i[0] + j[0] + non_abelian[3][G.index([i[1], j[1]])])%2, s(i[1], j[1])])

'''
Now we make a list order_two of elements of order two.  
'''

order_two = []

for i in EE:
    if i != [0, [0, 0]] and t(i, i) == [0, [0, 0]]:
        order_two.append(i)
        
print(len(order_two))

1
